In [1]:
import os
import boto3
from sagemaker import get_execution_role
from pprint import pprint
import json
import time
import pandas as pd

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# name of step function
str_name = 'step-genxii-pd-feat-select-boto3'

Project: 20231010-gen-xii


### Hyperparameter df

In [3]:
# make a dictionary of hyperparameters, save as df to s3, so I can re-convert it to dict in the images
dict_hyperparameters = {
    # data sets
    'STR_FILENAME_TRAIN': 'df_train_noleaks_pre.gzip',
    'STR_FILENAME_VALID': 'df_valid_noleaks_pre.gzip', # always use the full data set for the validation model
    # ITERATIONS - define once for consistency
    'INT_N_ITERATIONS': 1000,
    # proportion of iterations used for early stopping
    'PROP_EARLY_STOPPING': 0.05,
    # tuning - 1
    'INT_N_TUNING_JOBS_1': 100, # number of tuning jobs in the first tuning job
    # eval metric
    'STR_EVAL_METRIC': 'AUC',
}

# make df
df = pd.DataFrame(dict_hyperparameters.items(), columns=['keys','values'])

# save
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/02_pricing_pd/02_model/01_feat_select/05_step_function/{str_filename}'
df.to_csv(str_uri, index=False)

# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:275: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,keys,values
0,STR_FILENAME_TRAIN,df_train_noleaks_pre.gzip
1,STR_FILENAME_VALID,df_valid_noleaks_pre.gzip
2,INT_N_ITERATIONS,1000
3,PROP_EARLY_STOPPING,0.05
4,INT_N_TUNING_JOBS_1,100
5,STR_EVAL_METRIC,AUC


### Write ```definition.json```

In [4]:
%%writefile definition.json

{
  "Comment": "A description of my state machine",
  "StartAt": "GetStartingFeatures1",
  "States": {
    "GetStartingFeatures1": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-pd-starting-feats:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 2,
          "MaxAttempts": 6,
          "BackoffRate": 2
        }
      ],
      "Next": "Tuning1"
    },
    "Tuning1": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "tuning-1",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-pd-tuning-1-1:6",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-pd-tuning-1-1",
        "ArrayProperties": {
          "Size": "INT_N_TUNING_JOBS_1"
        }
      },
      "Next": "ConcatTuning1",
      "Retry": [
        {
          "ErrorEquals": [
            "States.ALL"
          ],
          "BackoffRate": 2,
          "IntervalSeconds": 1,
          "MaxAttempts": 5
        }
      ]
    },
    "ConcatTuning1": {
      "Type": "Task",
      "Resource": "arn:aws:states:::lambda:invoke",
      "OutputPath": "$.Payload",
      "Parameters": {
        "FunctionName": "arn:aws:lambda:us-west-2:836690756591:function:genxii-pd-concat-tuning-1:$LATEST"
      },
      "Retry": [
        {
          "ErrorEquals": [
            "Lambda.ServiceException",
            "Lambda.AWSLambdaException",
            "Lambda.SdkClientException",
            "Lambda.TooManyRequestsException"
          ],
          "IntervalSeconds": 2,
          "MaxAttempts": 6,
          "BackoffRate": 2
        }
      ],
      "Next": "FeatSelect1"
    },
    "FeatSelect1": {
      "Type": "Task",
      "Resource": "arn:aws:states:::batch:submitJob.sync",
      "Parameters": {
        "JobName": "feat-select",
        "JobDefinition": "arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-pd-feat-select-1:6",
        "JobQueue": "arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-pd-feat-select-1"
      },
      "End": true
    }
  }
}

Writing definition.json


### Make string definition

In [5]:
# load it
dict_definition = json.load(open('./definition.json'))
# make into string
str_definition = json.dumps(dict_definition)

# replace
str_definition = str_definition.replace('"INT_N_TUNING_JOBS_1"', str(dict_hyperparameters['INT_N_TUNING_JOBS_1']))

### Create state machine

In [6]:
cls_client_sfn = boto3.client('stepfunctions')

In [7]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [8]:
# list state machines
dict_response = cls_client_sfn.list_state_machines(
)
list_dict_state_machines = dict_response['stateMachines']
list_dict_state_names = [{dict_state_machine['name']: dict_state_machine['stateMachineArn']} for dict_state_machine in list_dict_state_machines]
dict_state_names = {key: val for dict_name in list_dict_state_names for key, val in dict_name.items()}
pprint(dict_state_names)

{'MyStateMachine-fldl4s6of': 'arn:aws:states:us-west-2:836690756591:stateMachine:MyStateMachine-fldl4s6of',
 'gen-xi-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xi-retro-scoring',
 'gen-xii-retro-scoring': 'arn:aws:states:us-west-2:836690756591:stateMachine:gen-xii-retro-scoring',
 'genxii-payload-parsing': 'arn:aws:states:us-west-2:836690756591:stateMachine:genxii-payload-parsing',
 'poc-step-genxii-lgd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-lgd-lambda-boto3',
 'poc-step-genxii-pd-lambda-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:poc-step-genxii-pd-lambda-boto3',
 'step-genxii-ad-feat-select-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-feat-select-boto3',
 'step-genxii-ad-model-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-model-boto3',
 'step-genxii-ad-pre-boto3': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-ad-pre-boto

In [9]:
# get list of just names
list_str_names = [list(dict_state_names.keys())[0] for dict_state_names in list_dict_state_names]
# if our name is in there
if str_name in list_str_names:
    print(f'State machine {str_name} exists, it will be deleted')
    str_arn = dict_state_names[str_name]
    print(f'Deleting {str_arn}')
    print('')
    dict_response = cls_client_sfn.delete_state_machine(
        stateMachineArn=str_arn,
    )
    pprint(dict_response)
else:
    print(f'State machine {str_name} does not exist, so it will not be deleted')

State machine step-genxii-pd-feat-select-boto3 exists, it will be deleted
Deleting arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-pd-feat-select-boto3

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Thu, 25 Apr 2024 19:42:18 GMT',
                                      'x-amzn-requestid': 'a9c76ce8-9dc5-4a77-90fe-464c10ac4a48'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'a9c76ce8-9dc5-4a77-90fe-464c10ac4a48',
                      'RetryAttempts': 0}}


In [10]:
# make a state machine
while True:
    try:
        dict_response = cls_client_sfn.create_state_machine(
            name=str_name,
            definition=str_definition,
            roleArn=str_role,
            type='STANDARD',
        )
        pprint(dict_response)
        break
    except:
        pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '137',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Thu, 25 Apr 2024 19:43:52 GMT',
                                      'x-amzn-error-pace-aws-throttling': 'true',
                                      'x-amzn-requestid': 'c70b259e-5f7e-438e-ab85-6b256984d108'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'c70b259e-5f7e-438e-ab85-6b256984d108',
                      'RetryAttempts': 3},
 'creationDate': datetime.datetime(2024, 4, 25, 19, 43, 52, 752000, tzinfo=tzlocal()),
 'stateMachineArn': 'arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-pd-feat-select-boto3'}


### Describe state machine

In [11]:
str_state_machine_arn = dict_response['stateMachineArn']
print(f'State Machine ARN: {str_state_machine_arn}')
dict_response = cls_client_sfn.describe_state_machine(
    stateMachineArn=str_state_machine_arn,
)
pprint(dict_response)

State Machine ARN: arn:aws:states:us-west-2:836690756591:stateMachine:step-genxii-pd-feat-select-boto3
{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '2503',
                                      'content-type': 'application/x-amz-json-1.0',
                                      'date': 'Thu, 25 Apr 2024 19:43:52 GMT',
                                      'x-amzn-requestid': '37064535-ead4-4019-b9b9-48501680244f'},
                      'HTTPStatusCode': 200,
                      'RequestId': '37064535-ead4-4019-b9b9-48501680244f',
                      'RetryAttempts': 0},
 'creationDate': datetime.datetime(2024, 4, 25, 19, 43, 52, 752000, tzinfo=tzlocal()),
 'definition': '{"Comment": "A description of my state machine", "StartAt": '
               '"GetStartingFeatures1", "States": {"GetStartingFeatures1": '
               '{"Type": "Task", "Resource": "arn:aws:states:::lambda:invoke", '
               '"Ou

### Execute step function workflow

In [12]:
# # start execution
# dict_response = cls_client_sfn.start_execution(
#     stateMachineArn=str_state_machine_arn,
# )

### Clean-up

In [13]:
os.remove('./definition.json')